## Setup

In [ ]:
import gc, torch

# Kill all model references from merge step
try:
    del model, base_model, peft_model, merged_model
except: pass

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# Verify VRAM is free
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {free/1e9:.2f}GB free / {total/1e9:.2f}GB total")

In [ ]:
!pip install -U bitsandbytes>=0.46.1

## Loading Model

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model_name = "/kaggle/input/models/somendrew/genz-merged-model/pytorch/default/1/genz-merged-model"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model.eval()
print("✅ Ready!")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Ready!


## Inferencing Function

In [3]:
def generate_genz(instruction, input_text=""):
    user_content = f"{instruction}\n\n{input_text}" if input_text.strip() else instruction
    
    messages = [{"role": "user", "content": user_content}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    response = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

## Test

In [4]:
tests = [
    ("Generate a sentence with complex vocabulary.", "Words: devious, antagonistic, ferocity"),
    ("Explain what black holes are.", ""),
    ("Give me 3 tips to stay productive.", ""),
    ("Rearrange this sentence to make it interesting.", "She left the party early."),
    ("Compose a haiku about a summer day.", ""),
]

for instruction, input_text in tests:
    print(f"\n📝 {instruction}")
    if input_text:
        print(f"📥 {input_text}")
    print(f"🔥 {generate_genz(instruction, input_text)}")
    print("-" * 60)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



📝 Generate a sentence with complex vocabulary.
📥 Words: devious, antagonistic, ferocity


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔥 The politician was known for his devious ways and antagonistic attitude towards his opponents. He spoke with ferocity during the debate, leaving the audience stunned.
------------------------------------------------------------

📝 Explain what black holes are.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔥 Black holes are super dense objects with gravity so strong that nothing can escape — not even light. They form when a star collapses in on itself, and they're basically cosmic vacuum cleaners sucking up everything around them. Cool but also kind of terrifying!
------------------------------------------------------------

📝 Give me 3 tips to stay productive.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔥 3 productivity hacks that actually work:
1. Prioritize like crazy — do the big stuff first so you don’t get swamped
2. Take breaks — short bursts of focus + rest = way more done than burning out
3. Delete distractions — block time-waste websites, tell Slack to buzz off for an hour. Focus mode only.
------------------------------------------------------------

📝 Rearrange this sentence to make it interesting.
📥 She left the party early.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🔥 Yo she bounced from that party wayyyy too early fr!
------------------------------------------------------------

📝 Compose a haiku about a summer day.
🔥 Warm sun, bright blue sky
Breeze hits different — no cap
Summer day straight vibes
------------------------------------------------------------
